# CEG-WM 端到端可执行验证 Notebook

## 目标

本 Notebook 用于生成
的可审计证据包，覆盖：

1. **模型加载**：Stable Diffusion 3.5 Medium（Transformer/DiT 架构，非 UNet）
2. **真实运行**：embed + detect 真实推理流程
3. **对齐验证**：paper_faithfulness 对齐报告必须 PASS
4. **测试验证**：pytest 全量测试通过
5. **门禁审计**：严格审计（--strict）必须 ALLOW_FREEZE
6. **证据打包**：可下载的完整 run_root + 审计报告 + 测试结果

## 输入要求

1. **项目 ZIP**：将整个 CEG-WM 仓库打包为 ZIP 上传
2. **Hugging Face Token**（可选）：若模型需要授权访问，请在下方 Cell 配置

## 输出产物

- `sd3_paper_faithfulness_run_bundle.zip`：包含 run_root、records、审计报告、测试日志
- `alignment_acceptance_summary.json`：对齐验收摘要

## A. 运行前准备：上传项目 ZIP

In [ ]:
# 克隆仓库到 Colab 环境
import os

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "Content_Chain"  # 包含 supported_models 声明的分支
REPO_NAME = "CEG-WM"
WORK_DIR = f"/content/{REPO_NAME}"

# 先切换到安全的目录（/content）
os.chdir("/content")
print(f"切换到临时目录: {os.getcwd()}")

# 如果目录已存在，先删除
if os.path.exists(WORK_DIR):
  print(f"清空已存在的目录: {WORK_DIR}")
  import shutil
  shutil.rmtree(WORK_DIR)
  print("✅ 已清空")

# 克隆仓库（从 Content_Chain 分支）
print(f"\n克隆仓库: {REPO_URL} (分支: {REPO_BRANCH})")
!git clone -b {REPO_BRANCH} {REPO_URL}

# 进入工作目录
os.chdir(WORK_DIR)
print(f"\n当前工作目录: {os.getcwd()}")
print(f"✅ 仓库克隆完成: {WORK_DIR} (源于分支: {REPO_BRANCH})")

## B. 安装依赖与环境固定

In [ ]:
# Cell B: 安装依赖
from pathlib import Path

print("=" * 60)
print("[B] 安装依赖包")
print("=" * 60)

# 检查 requirements.txt 是否存在
req_file = Path("requirements.txt")
if not req_file.exists():
    raise RuntimeError("未找到 requirements.txt 文件")

print("\n正在安装依赖（这可能需要几分钟）...")

# 安装核心依赖（使用仓库锁定版本）
!pip install -q --upgrade pip
!pip install -q torch==2.10.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers==0.32.0 transformers==4.45.2 accelerate==1.12.0
!pip install -q safetensors==0.7.0 pyyaml==6.0.2 pillow==11.2.1
!pip install -q numpy scipy pytest pydantic
!pip install -q huggingface_hub

print("\n✅ 依赖安装完成")

# 验证关键包版本
print("\n关键包版本:")
import torch
print(f"  - PyTorch: {torch.__version__}")
print(f"  - CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - CUDA 版本: {torch.version.cuda}")
    print(f"  - GPU 设备: {torch.cuda.get_device_name(0)}")

try:
    import diffusers
    print(f"  - Diffusers: {diffusers.__version__}")
except ImportError:
    print("  - Diffusers: ❌ 未安装")

try:
    import transformers
    print(f"  - Transformers: {transformers.__version__}")
except ImportError:
    print("  - Transformers: ❌ 未安装")

try:
    import safetensors
    print(f"  - Safetensors: {safetensors.__version__}")
except ImportError:
    print("  - Safetensors: ❌ 未安装")

## C. 配置 Hugging Face 凭据（可选）

In [ ]:
# 配置 HF_TONKEN

v = os.environ.get("HF_TOKEN")
print("HF_TOKEN exists:", v is not None)
print("HF_TOKEN length:", 0 if v is None else len(v))
print("HF_TOKEN head:", "" if v is None else v[:4] + "..." + v[-4:])

# 下载 Stable Diffusion 模型
print("="*80)
print("下载 Stable Diffusion 模型")
print("="*80)

from diffusers import DiffusionPipeline
from huggingface_hub import scan_cache_dir
import torch

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# 检查模型是否已缓存
def checkModelCached(modelId):
    try:
        cacheInfo = scan_cache_dir()
        for repo in cacheInfo.repos:
            if modelId in repo.repo_id:
                return True
        return False
    except:
        return False

if checkModelCached(MODEL_ID):
    print(f"模型已缓存: {MODEL_ID}")
    print("跳过下载")
else:
    print(f"模型未缓存 开始下载: {MODEL_ID}")
    print("这可能需要几分钟时间...")

    try:
        # 下载模型但不加载到内存
        _ = DiffusionPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("模型下载完成")
    except Exception as e:
        print(f"模型下载失败: {e}")
        print("尝试继续执行...")

# 清理内存
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("模型准备完成")

## D. 仓库自检

In [ ]:
# Cell D: 仓库自检
import sys

print("=" * 60)
print("[D] 仓库自检")
print("=" * 60)

# 语法检查
print("\n[1/3] 编译检查 main/ 模块...")
!python -m compileall main/ 2>&1 | tail -5

# 导入检查
print("\n[2/3] 测试导入核心模块...")
try:
    sys.path.insert(0, os.getcwd())
    import main
    from main.core import config_loader, digests, records_io
    from main.diffusion.sd3 import pipeline_factory
    print("✅ 核心模块导入成功")
except Exception as e:
    print(f"❌ 导入失败: {e}")
    raise

# 打印仓库版本信息
print("\n[3/3] 仓库冻结契约版本:")
try:
    from main.core.contracts import load_frozen_contracts
    contracts = load_frozen_contracts("configs/frozen_contracts.yaml")
    print(f"  - contract_version: {contracts.contract_version}")
    print(f"  - contract_digest: {contracts.contract_digest[:16]}...")
    print(f"  - contract_bound_digest: {contracts.contract_bound_digest[:16]}...")
except Exception as e:
    print(f"⚠️  无法读取冻结契约: {e}")

print("\n✅ 仓库自检完成")

## E. 准备运行配置

In [ ]:
# Cell E: 定义本地辅助函数
import yaml
from pathlib import Path
from typing import Dict, Any


def torch_cuda_available() -> bool:
    """检查 CUDA 是否可用。"""
    try:
        import torch
        return torch.cuda.is_available()
    except:
        return False


def generate_embed_config(
    output_path: str,
    model_id: str = "stabilityai/stable-diffusion-3.5-medium",
    model_source: str = "hf",
    hf_revision: str = "main",
    inference_prompt: str = "a serene mountain landscape with a lake",
    inference_num_steps: int = 20,
    inference_guidance_scale: float = 4.5,
    seed: int = 42,
    enable_content: bool = True,
    enable_geometry: bool = False,
    enable_paper_faithfulness: bool = True,
    enable_pipeline_inference: bool = True,
    enable_trajectory_tap: bool = True,
    enable_real_inference: bool = True,
    use_fp16: bool = True,
) -> Dict[str, Any]:
    """
    生成 embed 运行配置文件（完整的真实实现配置）。

    Generate and save runtime configuration for embed stage with real inference enabled.

    Args:
        output_path: Output YAML file path.
        model_id: Hugging Face model ID.
        model_source: Model source ('hf' or 'local').
        hf_revision: Hugging Face revision (branch/tag/commit hash).
        inference_prompt: Text prompt for inference.
        inference_num_steps: Number of diffusion steps (20-28 recommended for SD3).
        inference_guidance_scale: Guidance scale (4.5-7.0 recommended for SD3).
        seed: Random seed for reproducibility.
        enable_content: Enable content chain watermarking.
        enable_geometry: Enable geometry chain watermarking.
        enable_paper_faithfulness: Enable paper faithfulness alignment check.
        enable_pipeline_inference: Enable P1 (pipeline fingerprint).
        enable_trajectory_tap: Enable P2 (trajectory digest).
        enable_real_inference: Enable real SD3 inference (non-placeholder).
        use_fp16: Use float16 for memory efficiency in Colab.

    Returns:
        Configuration dict.
    """
    if not isinstance(output_path, str) or not output_path:
        raise ValueError("output_path must be non-empty string")
    if not isinstance(model_id, str) or not model_id:
        raise ValueError("model_id must be non-empty string")
    if model_source not in ("hf", "local"):
        raise ValueError("model_source must be 'hf' or 'local'")
    if inference_num_steps < 1 or inference_num_steps > 100:
        raise ValueError("inference_num_steps must be in range [1, 100]")
    if inference_guidance_scale < 0:
        raise ValueError("inference_guidance_scale must be non-negative")

    # 【关键】当启用 paper_faithfulness 时，必须强制启用所有必要的真实实现参数
    actual_inference_enabled = enable_pipeline_inference and enable_real_inference
    actual_tap_enabled = enable_trajectory_tap
    if enable_paper_faithfulness:
        actual_inference_enabled = True
        actual_tap_enabled = True

    runtime_config = {
        "pipeline_impl_id": "sd3_diffusers_real_v1",  # 【关键】使用真实 diffusers 实现，非 shell
        "pipeline_build_enabled": True,  # 【强制】必须构造真实 pipeline
        "inference_enabled": actual_inference_enabled,  # 【强制】启用真实推理
        "trajectory_tap": {
            "enabled": actual_tap_enabled,
            "sample_at_steps": [5, 10, 15, 20],
            "sample_layer_names": ["transformer"],
        },
        "model_id": model_id,
        "model_source": model_source,
        "hf_revision": hf_revision,
        "policy_path": "content_only",
        "target_fpr": 0.01,
        "device": "cuda" if torch_cuda_available() else "cpu",  # 自动检测 CUDA
        "detect": {
            "content": {"enabled": enable_content},
            "geometry": {"enabled": enable_geometry}
        },
        "watermark": {
            "key_id": "master_key_001",
            "pattern_id": "pattern_standard_v1",
            "strength": 0.8,
            "plan_digest": None,
            "lf": {
                "enabled": True,
                "codebook_id": "lf_codebook_v1",
                "ecc": "sparse_ldpc",
                "ecc_sparsity": 3,
                "strength": 0.5,
                "variance": 1.5,
            },
            "hf": {
                "enabled": True,
                "codebook_id": "hf_codebook_v1",
                "ecc": 2,
                "tail_truncation_ratio": 0.1,
                "tail_truncation_mode": "top_k_per_latent",
                "tau": 1.0,
            },
            "subspace": {
                "enabled": True,
                "frame": "latent",
                "selector_id": "selector_dct_v1",
                "grid_rows": 4,
                "grid_cols": 4,
                "grid_h": 64,
                "grid_w": 64,
                "k": 512,
                "topk": 128,
                "score_type": "amplitude",
                "channel_agg": "sum",
                "rank": 8,
                "energy_ratio": 0.9,
                "mask_digest_binding": True,
            },
            "mask": {
                "threshold": 0.5,
                "resolution_binding": "512x512",
                "postprocess": {
                    "enabled": True,
                    "kernel": "gaussian",
                },
            },
            "injection_site_spec": {
                "hook_type": "callback_on_step_end",
                "target_module_name": "StableDiffusion3Pipeline",
                "target_tensor_name": "latents",
                "hook_timing": "after_scheduler_step",
                "injection_rule_digest": "placeholder",
            },
        },
        "inference_prompt": inference_prompt,
        "inference_num_steps": inference_num_steps,
        "inference_guidance_scale": inference_guidance_scale,
        "inference_height": 512,
        "inference_width": 512,
        "generation": {
            "num_inference_steps": inference_num_steps,
            "guidance_scale": inference_guidance_scale,
            "prompt": inference_prompt,
            "seed": seed,
        },
        "model": {
            "height": 512,
            "width": 512,
            "dtype": "float16" if use_fp16 else "float32",
        },
        "enable_xformers": True,
        "evaluate": {
            "decoder_type": "content_correlation",
            "target_fpr": 0.01,
        },
        "impl": {
            "content_extractor_id": "content_baseline_noop_v1",
            "geometry_extractor_id": "geometry_baseline_noop_v1",
            "fusion_rule_id": "fusion_baseline_identity_v1",
            "subspace_planner_id": "subspace_baseline_full_v1",
            "sync_module_id": "geometry_sync_baseline_v1",
        },
    }

    # 【关键】添加 paper_faithfulness 配置
    if enable_paper_faithfulness:
        runtime_config["paper_faithfulness"] = {
            "enabled": True,
            "alignment_check": True,
        }

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        yaml.dump(runtime_config, f, default_flow_style=False, allow_unicode=True)

    return runtime_config


def print_config_summary(cfg: Dict[str, Any], config_path: str) -> None:
    """打印配置摘要，突出真实实现参数。"""
    print("\n" + "=" * 80)
    print("[CONFIG] Embed 运行配置已生成 (真实实现模式)")
    print("=" * 80)

    print(f"\n[文件路径]")
    print(f"  {config_path}")
    print(f"  大小: {Path(config_path).stat().st_size} 字节")

    print(f"\n[模型参数]")
    print(f"  - model_id: {cfg.get('model_id')}")
    print(f"  - model_source: {cfg.get('model_source')}")
    print(f"  - hf_revision: {cfg.get('hf_revision')}")
    print(f"  - device: {cfg.get('device', 'cpu')}")
    print(f"  - dtype: {cfg.get('model', {}).get('dtype', 'float32')}")

    print(f"\n[推理参数]")
    print(f"  - inference_prompt: {cfg.get('inference_prompt')}")
    print(f"  - inference_num_steps: {cfg.get('inference_num_steps')}")
    print(f"  - inference_guidance_scale: {cfg.get('inference_guidance_scale')}")

    print(f"\n[真实实现启用标志]")
    print(f"  ✅ pipeline_build_enabled: {cfg.get('pipeline_build_enabled')}")
    print(f"  ✅ inference_enabled: {cfg.get('inference_enabled')}")
    
    traj = cfg.get('trajectory_tap', {})
    print(f"  ✅ trajectory_tap.enabled: {traj.get('enabled')}")

    print(f"\n[Paper Faithfulness 证据配置]")
    pf = cfg.get("paper_faithfulness", {})
    if pf.get("enabled"):
        print(f"  ✅ Paper Faithfulness: 启用")
        inf_en = cfg.get("inference_enabled")
        traj_en = cfg.get("trajectory_tap", {}).get("enabled")
        print(f"    [P1] Pipeline 推理: {'✅ 启用' if inf_en else '❌ 禁用'}")
        print(f"    [P2] 轨迹采样: {'✅ 启用' if traj_en else '❌ 禁用'}")
    else:
        print(f"  ❌ Paper Faithfulness: 禁用")

    print(f"\n[水印链配置]")
    watermark = cfg.get("watermark", {})
    print(f"  - LF（PRC）: {'✅' if watermark.get('lf', {}).get('enabled') else '❌'}")
    print(f"  - HF（T2SMark）: {'✅' if watermark.get('hf', {}).get('enabled') else '❌'}")
    print(f"  - 子空间: {'✅' if watermark.get('subspace', {}).get('enabled') else '❌'}")
    print(f"\n✅ 真实实现配置准备完成\n")

print("✅ 辅助函数已定义")


In [ ]:
# Cell F: 定义运行路径 + 生成配置（真实实现版本）
import os
from pathlib import Path

print("=" * 60)
print("[F] 准备运行环境（真实 Embed 实现）")
print("=" * 60)

# 定义运行相关的路径
RUN_ROOT = Path("/content/CEG-WM/runs/sd3_real_inference_run")
LOGS_DIR = Path("/content/CEG-WM/logs")
TEMP_CONFIG = Path("/content/temp_runtime_real.yaml")

LOGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[1/2] 运行路径配置:")
print(f"  - RUN_ROOT: {RUN_ROOT}")
print(f"  - LOGS_DIR: {LOGS_DIR}")
print(f"  - TEMP_CONFIG: {TEMP_CONFIG}")

# 生成配置（真实推理版本）
print(f"\n[2/2] 生成 Embed 运行配置 (真实实现模式)...")
print("ℹ️  使用优化配置：25 步推理 + float16 + 完整对齐检查\n")

cfg = generate_embed_config(
    output_path=str(TEMP_CONFIG),
    model_id="stabilityai/stable-diffusion-3.5-medium",
    model_source="hf",
    hf_revision="main",
    inference_prompt="a beautiful landscape with mountains and lake",
    inference_num_steps=25,  # 25 步用于稳定的轨迹采样
    inference_guidance_scale=4.5,
    seed=42,
    enable_content=True,
    enable_geometry=False,
    enable_paper_faithfulness=True,  # 【关键】启用 paper faithfulness
    enable_pipeline_inference=True,   # 【P1】Pipeline fingerprint
    enable_trajectory_tap=True,       # 【P2】Trajectory digest
    enable_real_inference=True,       # 【关键】强制启用真实推理（非 placeholder）
    use_fp16=True,                    # 使用 float16 以节省内存
)

print_config_summary(cfg, str(TEMP_CONFIG))

# 定义全局变量供后续 cell 使用
SEED = 42
INFERENCE_NUM_STEPS = 25
INFERENCE_GUIDANCE_SCALE = 4.5
INFERENCE_PROMPT = "a beautiful landscape with mountains and lake"
ENABLE_PAPER_FAITHFULNESS = True
ENABLE_REAL_INFERENCE = True
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

print("[配置参数确认]")
print(f"  ✓ SEED: {SEED}")
print(f"  ✓ INFERENCE_NUM_STEPS: {INFERENCE_NUM_STEPS}")
print(f"  ✓ INFERENCE_GUIDANCE_SCALE: {INFERENCE_GUIDANCE_SCALE}")
print(f"  ✓ ENABLE_PAPER_FAITHFULNESS: {ENABLE_PAPER_FAITHFULNESS}")
print(f"  ✓ ENABLE_REAL_INFERENCE: {ENABLE_REAL_INFERENCE}")


## G. 执行 Embed 并验证对齐

In [ ]:
# Cell G: 执行真实 Embed 并验证对齐（真实推理版本）
import subprocess
import json
import shutil
from pathlib import Path
import yaml

print("=" * 60)
print("[G] 执行真实 Embed 推理并验证对齐")
print("=" * 60)

run_root_str = str(RUN_ROOT)
logs_dir_str = str(LOGS_DIR)
temp_config_str = str(TEMP_CONFIG)

# 【诊断】检查配置文件中的关键字段
print(f"\n[预检查] 验证配置文件...")
try:
    with open(temp_config_str, "r") as f:
        config_content = yaml.safe_load(f)
    
    print(f"  ✅ 配置文件加载成功")
    
    # 【关键】验证 pipeline_impl_id
    pipeline_impl_id = config_content.get('pipeline_impl_id')
    print(f"    - pipeline_impl_id: {pipeline_impl_id}")
    if pipeline_impl_id != "sd3_diffusers_real_v1":
        print(f"      ⚠️  警告: 应为 'sd3_diffusers_real_v1'，当前为 '{pipeline_impl_id}'")
    
    print(f"    - inference_enabled: {config_content.get('inference_enabled')}")
    print(f"    - pipeline_build_enabled: {config_content.get('pipeline_build_enabled')}")
    print(f"    - device: {config_content.get('device')}")
    print(f"    - dtype: {config_content.get('model', {}).get('dtype')}")
    
    pf_cfg = config_content.get("paper_faithfulness", {})
    print(f"    - paper_faithfulness.enabled: {pf_cfg.get('enabled')}")
    print(f"    - paper_faithfulness.alignment_check: {pf_cfg.get('alignment_check')}")
    
    traj_cfg = config_content.get("trajectory_tap", {})
    print(f"    - trajectory_tap.enabled: {traj_cfg.get('enabled')}")
    
    # 验证必需的真实实现参数
    errors = []
    if not config_content.get('inference_enabled'):
        errors.append("inference_enabled 为 False，推理将被跳过")
    if not config_content.get('pipeline_build_enabled'):
        errors.append("pipeline_build_enabled 为 False，pipeline 不会被构建")
    if not pipeline_impl_id:
        errors.append("pipeline_impl_id 缺失，将使用默认 shell 实现")
    elif pipeline_impl_id == "sd3_diffusers_shell_v1":
        errors.append("pipeline_impl_id 为 shell 实现，将返回 None")
    if not (pf_cfg.get('enabled')):
        errors.append("paper_faithfulness 未启用，对齐检查将被跳过")
    
    if errors:
        print(f"\n  ⚠️  配置问题:")
        for err in errors:
            print(f"      - {err}")
        raise ValueError("配置验证失败，请检查上述问题")
    
except Exception as e:
    print(f"  ❌ 配置文件检查失败: {e}")
    raise

if Path(run_root_str).exists():
    print(f"\n清空已存在的 run_root 目录: {run_root_str}")
    shutil.rmtree(run_root_str)
    print("✅ 已清空")

Path(run_root_str).mkdir(parents=True, exist_ok=True)

# 【关键】embed 命令（无需额外参数，真实实现已在配置中启用）
cmd = [
    "python", "-m", "main.cli.run_embed",
    "--out", run_root_str,
    "--config", temp_config_str
]

print("\n执行命令:")
print(" ".join(cmd))

print(f"\n【说明】此次执行将使用真实 SD3 推理，而不是 placeholder 模式:")
print(f"  - pipeline_impl_id=sd3_diffusers_real_v1: 使用真实 diffusers 实现")
print(f"  - pipeline_build_enabled=True: SD3 pipeline 将被完整构建")
print(f"  - inference_enabled=True: 真实的扩散推理将执行")
print(f"  - trajectory_tap.enabled=True: 轨迹采样将记录推理过程")
print(f"  - paper_faithfulness.enabled=True: 对齐检查将验证证据")

embed_log = Path(logs_dir_str) / "embed_real.log"
print(f"\n日志输出: {embed_log}")
print("\n开始执行真实推理...\n")

with open(embed_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n❌ Embed 执行失败，退出码: {proc.returncode}")
    print("\n【故障排查】查看错误详情...\n")
    try:
        with open(embed_log, "r") as f:
            log_content = f.read()
        # 输出最后 150 行
        lines = log_content.split('\n')
        error_section = '\n'.join(lines[-150:])
        print(error_section)
    except:
        !tail -150 {embed_log}
    
    print("\n【诊断建议】")
    print("1️⃣  读取日志以发现具体错误（如 CUDA OOM、模型加载失败等）")
    print("2️⃣  如果是内存不足，请尝试减少 inference_num_steps 或使用量化")
    print("3️⃣  检查 Hugging Face 模型是否可访问（需要网络）")
    print("4️⃣  确认 PyTorch 和相关库已正确安装")
    
    raise RuntimeError(f"Embed 执行失败 (退出码: {proc.returncode})")

print("\n✅ Embed 执行成功")

records_dir = Path(run_root_str) / "records"
embed_record_files = list(records_dir.glob("embed_*.json"))

if not embed_record_files:
    print("\n⚠️  未找到 embed_record.json")
    !ls -lh {records_dir} 2>/dev/null || echo "records/ 目录不存在"
    embed_record_path = None
else:
    embed_record_path = embed_record_files[0]
    print(f"\n找到 embed record: {embed_record_path.name}")
    
    with open(embed_record_path, "r") as f:
        embed_record = json.load(f)
    
    print("\n" + "=" * 80)
    print("=== 真实推理执行结果 ===")
    print("=" * 80)
    
    # 推理状态
    print("\n[推理执行状态]")
    inference_status = embed_record.get("inference_status", "unknown")
    print(f"  - inference_status: {inference_status}")
    
    if inference_status == "failed":
        inference_error = embed_record.get("inference_error", "未知错误")
        print(f"  - inference_error: {inference_error}")
    
    # 生成结果
    print("\n[生成结果]")
    artifacts_dir = Path(run_root_str) / "artifacts"
    if artifacts_dir.exists():
        image_files = list(artifacts_dir.glob("*.png")) + list(artifacts_dir.glob("*.jpg"))
        if image_files:
            print(f"  ✅ 生成了 {len(image_files)} 张图像:")
            for img in image_files[:3]:  # 只显示前 3 张
                print(f"     - {img.name}")
        else:
            print(f"  ⚠️  artifacts/ 目录存在但无图像文件")
            if inference_status == "failed":
                print(f"     原因: 推理失败 ({inference_status})")
            !ls -lh {artifacts_dir}
    else:
        print(f"  ❌ artifacts/ 目录不存在")
    
    # Paper Faithfulness
    print("\n[Paper Faithfulness]")
    pf_spec = embed_record.get("paper_faithfulness_spec_digest", "未提供")
    print(f"  - spec_digest: {pf_spec[:20]}...")
    
    pf_enabled = embed_record.get("paper_faithfulness_enabled", False)
    print(f"  - enabled: {pf_enabled}")
    
    # 内容证据
    print("\n[内容证据]")
    alignment_report = embed_record.get("alignment_report", {})
    overall_status = alignment_report.get("overall_status", "未知")
    print(f"  - alignment_report.overall_status: {overall_status}")
    
    alignment_digest = alignment_report.get("alignment_digest", "未知")[:20]
    print(f"  - alignment_digest: {alignment_digest}...")
    
    pipeline_fp = embed_record.get("pipeline_fingerprint_digest", "未知")[:20]
    print(f"  - pipeline_fingerprint_digest: {pipeline_fp}...")
    
    injection_site = embed_record.get("injection_site_digest", "未知")[:20]
    print(f"  - injection_site_digest: {injection_site}...")
    
    if overall_status == "PASS":
        print(f"\n✅ 对齐验证状态: {overall_status}")
    else:
        print(f"\n⚠️  对齐验证状态: {overall_status}")
    
    # 对齐检查详情
    checks = alignment_report.get("checks", [])
    if checks:
        print(f"\n[对齐检查详情] ({len(checks)} 项):")
        for i, check in enumerate(checks, 1):
            check_name = check.get("check_name", "未知")
            status = check.get("status", "未知")
            rule = check.get("rule", "未说明")
            reason = check.get("reason", "未说明")
            
            status_icon = "✅" if status == "PASS" else "❌"
            print(f"\n  [{i}] {status_icon} {check_name}: {status}")
            print(f"      规则: {rule}")
            if status != "PASS":
                print(f"      原因: {reason}")


In [ ]:
# [G-Diagnostic] 深度诊断 Pipeline 和推理错误
import json
from pathlib import Path

print("=" * 80)
print("[G-Diagnostic] 深度诊断：Pipeline 和推理失败原因")
print("=" * 80)

embed_log = Path(LOGS_DIR) / "embed.log"

if embed_log.exists():
    with open(embed_log, "r") as f:
        full_log = f.read()
    
    print("\n[1] embed.log 完整内容（用于深度诊断）:")
    print("-" * 80)
    print(full_log)
    print("-" * 80)
    
    # 搜索关键错误信息
    print("\n[2] 关键错误搜索:")
    
    error_keywords = [
        "pipeline",
        "Pipeline",
        "inference",
        "Inference",
        "FAILED",
        "ERROR",
        "error",
        "Exception",
        "Traceback",
        "paper_faithful",
    ]
    
    for keyword in error_keywords:
        if keyword in full_log:
            count = full_log.count(keyword)
            if count > 0:
                print(f"  - '{keyword}' 出现 {count} 次")
    
    # 特别检查是否有 "placeholder" 或 "mock" 模式的迹象
    if "placeholder" in full_log.lower():
        print(f"\n[关键发现] 检测到 'placeholder' 模式")
        print(f"  这表明 embed 可能在使用 mock/占位符 执行，而非真实推理")
        print(f"  解决方案：")
        print(f"    - 检查是否有 --enable-actual-inference 或类似参数")
        print(f"    - 或检查环境变量设置")
    
    # 检查 paper_faithfulness 相关的日志
    print(f"\n[3] Paper Faithfulness 相关日志:")
    lines = full_log.split('\n')
    for i, line in enumerate(lines):
        if 'paper' in line.lower() or 'faithful' in line.lower():
            print(f"  Line {i}: {line}")
    
    # 检查 inference 相关日志
    print(f"\n[4] Inference 相关日志:")
    for i, line in enumerate(lines):
        if 'inference' in line.lower() or 'infer' in line.lower():
            print(f"  Line {i}: {line}")
else:
    print(f"❌ embed.log 不存在: {embed_log}")

# 检查 embed_record.json 中是否有额外的诊断信息
print(f"\n[5] Embed Record 完整诊断信息:")
if embed_record_path and embed_record_path.exists():
    with open(embed_record_path, "r") as f:
        embed_rec_full = json.load(f)
    
    # 打印 paper_faithfulness 完整内容
    pf_full = embed_rec_full.get("paper_faithfulness", {})
    if pf_full:
        print(f"\n[paper_faithfulness 完整内容]")
        print(json.dumps(pf_full, indent=2, ensure_ascii=False))
    
    # 打印 execution_report（如果存在）
    exec_report = embed_rec_full.get("execution_report", {})
    if exec_report:
        print(f"\n[execution_report]")
        print(json.dumps(exec_report, indent=2, ensure_ascii=False))
    
    # 打印所有顶级字段
    print(f"\n[所有顶级字段]")
    for key in sorted(embed_rec_full.keys()):
        val_type = type(embed_rec_full[key]).__name__
        print(f"  - {key}: {val_type}")


## G-Real: 深度诊断 - Pipeline 构建失败根因

**目的**：分析为什么 `pipeline_status` 是 `None` 而不是预期的状态值

**发现**：
- ✅ 配置文件：`pipeline_build_enabled=True`, `paper_faithfulness.enabled=True`  
- ❌ 实际结果：`pipeline_status=None`, `pipeline_obj=None`
- ⚠️ 怀疑：`build_pipeline_shell()` 可能根本没有被正确调用或返回了异常值

**下一步**：直接测试 `pipeline_factory.build_pipeline_shell()` 的行为


In [ ]:
# Cell G-Real: 直接测试 Pipeline Factory
import yaml
import sys
from pathlib import Path

print("=" * 80)
print("[G-Real] 直接测试 Pipeline Factory 行为")
print("=" * 80)

# 加载配置
config_path = Path("/content/temp_runtime_real.yaml")
if not config_path.exists():
    print(f"❌ 配置文件不存在: {config_path}")
    print("请先运行 Cell F 生成配置")
else:
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    print("\n[1] 导入 pipeline_factory 模块...")
    try:
        from main.diffusion.sd3 import pipeline_factory
        print("  ✅ 导入成功")
    except Exception as e:
        print(f"  ❌ 导入失败: {e}")
        import traceback
        traceback.print_exc()
        raise
    
    print("\n[2] 检查配置参数...")
    print(f"  - pipeline_build_enabled: {cfg.get('pipeline_build_enabled')}")
    print(f"  - model_id: {cfg.get('model_id')}")
    print(f"  - model_source: {cfg.get('model_source')}")
    print(f"  - device: {cfg.get('device')}")
    
    pf_cfg = cfg.get('paper_faithfulness', {})
    print(f"  - paper_faithfulness.enabled: {pf_cfg.get('enabled')}")
    
    print("\n[3] 调用 build_pipeline_shell(cfg)...")
    try:
        pipeline_result = pipeline_factory.build_pipeline_shell(cfg)
        print(f"  ✅ 调用成功")
        print(f"  返回类型: {type(pipeline_result)}")
        print(f"  返回值是否为 None: {pipeline_result is None}")
    except Exception as e:
        print(f"  ❌ 调用失败: {e}")
        import traceback
        traceback.print_exc()
        raise
    
    if pipeline_result is not None:
        print(f"\n[4] 检查返回结果字段 ({len(pipeline_result)} 个):")
        for key in sorted(pipeline_result.keys()):
            value = pipeline_result[key]
            if key == "pipeline_obj":
                obj_type = type(value).__name__ if value is not None else "None"
                print(f"    - {key}: <{obj_type}>")
            elif isinstance(value, str) and len(value) > 60:
                print(f"    - {key}: '{value[:60]}...'")
            elif isinstance(value, dict):
                print(f"    - {key}: <dict with {len(value)} keys>")
            else:
                print(f"    - {key}: {value}")
        
        print("\n[5] 关键字段详细分析:")
        print("-" * 80)
        
        pipeline_status = pipeline_result.get('pipeline_status')
        pipeline_error = pipeline_result.get('pipeline_error')
        pipeline_obj = pipeline_result.get('pipeline_obj')
        
        print(f"\n  pipeline_status: {pipeline_status}")
        print(f"  pipeline_error: {pipeline_error}")
        print(f"  pipeline_obj: {pipeline_obj is not None}")
        
        if pipeline_status is None:
            print("\n  ❌ 严重错误: pipeline_status 为 None!")
            print("     这表明 build_pipeline_shell() 实现有 bug")
            print("     expected: 'unbuilt', 'built', or 'failed'")
            print("     actual: None")
        elif pipeline_status == "unbuilt":
            print("\n  ⚠️  Pipeline 状态: 未构建 (unbuilt)")
            print("     这意味着 _resolve_pipeline_build_enabled() 返回了 False")
            print("\n  尝试直接调用 _resolve_pipeline_build_enabled()...")
            try:
                # 通过反射访问私有函数
                resolve_func = getattr(pipeline_factory, '_resolve_pipeline_build_enabled', None)
                if resolve_func:
                    enabled, error = resolve_func(cfg)
                    print(f"    enabled: {enabled}")
                    print(f"    error: {error}")
                    if not enabled:
                        print(f"\n    ❌ 构建被禁用!")
                        print(f"       理由: {error}")
                else:
                    print("    ⚠️  无法访问私有函数")
            except Exception as e:
                print(f"    ❌ 调用失败: {e}")
        elif pipeline_status == "failed":
            print("\n  ❌ Pipeline 构建失败")
            print(f"     错误信息: {pipeline_error}")
            
            if "model_id" in str(pipeline_error):
                print("\n  [诊断建议] 模型 ID 相关错误")
                print("    - 检查 model_id 是否正确")
                print("    - 确认 HuggingFace 可访问")
            elif "CUDA" in str(pipeline_error) or "memory" in str(pipeline_error).lower():
                print("\n  [诊断建议] CUDA/内存相关错误")
                print("    - 尝试使用 CPU: device='cpu'")
                print("    - 减少 batch size")
                print("    - 启用模型量化")
            elif "import" in str(pipeline_error).lower():
                print("\n  [诊断建议] 导入错误")
                print("    - 检查 diffusers, transformers 是否安装")
                print("    - 运行 Cell B 重新安装依赖")
        elif pipeline_status == "built":
            print("\n  ✅ Pipeline 构建成功!")
            if pipeline_obj is None:
                print("     但 pipeline_obj 为 None（这很异常）")
            else:
                print(f"     pipeline_obj 类型: {type(pipeline_obj).__name__}")
                print(f"     可以进行推理")
    else:
        print("\n  ❌ 严重错误: build_pipeline_shell() 返回 None")
        print("     这不应该发生，函数应该总是返回 dict")
    
    print("\n" + "=" * 80)
    print("[诊断总结]")
    print("=" * 80)
    
    if pipeline_result and pipeline_result.get('pipeline_status') == 'built':
        print("\n✅ Pipeline 构建成功，可以进行真实推理")
    elif pipeline_result and pipeline_result.get('pipeline_status') == 'failed':
        print("\n❌ Pipeline 构建失败，需要修复错误后重试")
        print(f"   错误: {pipeline_result.get('pipeline_error')}")
    elif pipeline_result and pipeline_result.get('pipeline_status') == 'unbuilt':
        print("\n⚠️  Pipeline 未构建，需要检查配置")
        print("   可能原因：配置标志未正确传递")
    elif pipeline_result and pipeline_result.get('pipeline_status') is None:
        print("\n❌ Pipeline 状态为 None，这是代码 bug")
        print("   需要修复 build_pipeline_shell() 的实现")
    else:
        print("\n❌ 无法获取 pipeline_result")


In [ ]:
# [G-Parameters] 探索 embed 命令支持的参数和模式
import subprocess
from pathlib import Path

print("=" * 80)
print("[G-Parameters] 探索 CLI 参数和环境变量")
print("=" * 80)

# 尝试获取 embed 命令的帮助
print("\n[1] 获取 embed 命令的帮助信息:")
print("-" * 80)

try:
    result = subprocess.run(
        ["python", "-m", "main.cli.run_embed", "--help"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
except Exception as e:
    print(f"❌ 获取帮助失败: {e}")

# 检查是否有环境变量可以配置推理模式
print("\n[2] 环境变量检查:")
print("-" * 80)

env_vars = {
    "CEG_WM_ENABLE_INFERENCE": os.environ.get("CEG_WM_ENABLE_INFERENCE"),
    "CEG_WM_ENABLE_TORCH": os.environ.get("CEG_WM_ENABLE_TORCH"),
    "CEG_WM_INFERENCE_MODE": os.environ.get("CEG_WM_INFERENCE_MODE"),
    "CEG_WM_PLACEHOLDER_MODE": os.environ.get("CEG_WM_PLACEHOLDER_MODE"),
    "TORCH_ENABLE": os.environ.get("TORCH_ENABLE"),
}

for var, value in env_vars.items():
    print(f"  {var}: {value if value is not None else '(未设置)'}")

# 尝试检查配置加载器是否有相关说明
print("\n[3] 检查配置加载器中的推理模式:")
print("-" * 80)

try:
    from main.core import config_loader
    import inspect
    
    # 获取 config_loader 的源码或文档
    if hasattr(config_loader, '__doc__'):
        print(f"Module doc: {config_loader.__doc__}")
    
    # 列出公共函数
    public_funcs = [name for name in dir(config_loader) if not name.startswith('_')]
    print(f"\nPublic functions/classes: {public_funcs[:10]}")  # 只显示前 10 个
    
except Exception as e:
    print(f"⚠️  无法检查配置加载器: {e}")

print("\n[问题总结]")
print("-" * 80)
print("""
根据诊断信息，embed 可能处于以下状态之一：

1. 【最可能】使用了 placeholder/mock 模式
   - 原因：日志显示"Generating embed record (placeholder)"
   - 解决：可能需要显式启用真实推理
   
2. 【次可能】配置未正确传递给 embed 引擎
   - 原因：配置预检查通过，但 embed_record 中 enabled=False
   - 解决：检查配置文件中是否有额外的标志需要设置
   
3. 【技术问题】Pipeline 对象初始化失败
   - 原因：pipeline_obj_is_none 和 inference_failed
   - 解决：查看完整 embed.log 获取具体的初始化错误

【后续步骤】
1. 运行上一个诊断 Cell 查看完整的 embed.log 内容
2. 基于 log 内容，尝试以下方案之一：
   a) 在 embed 命令中添加参数（例如：--real-inference, --enable-torch 等）
   b) 设置环境变量后重新运行
   c) 修改配置文件中的其他字段（如果 log 中有提示）
""")


## H. 执行 Detect 并生成审计证据

In [ ]:
# Cell H: 执行 Detect + 验证结构 + 生成摘要对照表
import subprocess
import json
import shutil
from pathlib import Path
import pandas as pd

print("=" * 80)
print("[H] 执行 Detect 并生成审计证据")
print("=" * 80)

if embed_record_path is None:
    print("\n⚠️  跳过 detect（embed_record 未找到）")
else:
    # === [1/3] 执行 Detect ===
    print("\n" + "=" * 60)
    print("[H-1] 执行 Detect")
    print("=" * 60)
    
    detect_run_root = RUN_ROOT + "_detect"
    
    if Path(detect_run_root).exists():
        print(f"\n清空已存在的 detect run_root 目录: {detect_run_root}")
        shutil.rmtree(detect_run_root)
        print("✅ 已清空")
    
    Path(detect_run_root).mkdir(parents=True, exist_ok=True)
    
    cmd = [
        "python", "-m", "main.cli.run_detect",
        "--out", detect_run_root,
        "--config", "configs/default.yaml",
        "--input", str(embed_record_path)
    ]
    
    print("\n执行命令:")
    print(" ".join(cmd[:8]) + " ...")
    
    detect_log = Path(LOGS_DIR) / "detect.log"
    print(f"\n日志输出: {detect_log}")
    print("\n开始执行...\n")
    
    with open(detect_log, "w") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
        
        proc.wait()
    
    if proc.returncode != 0:
        print(f"\n❌ Detect 失败，退出码: {proc.returncode}")
        print("\n最后 50 行日志:")
        !tail -50 {detect_log}
    else:
        print("\n✅ Detect 执行成功")
        
        detect_records_dir = Path(detect_run_root) / "records"
        detect_record_files = list(detect_records_dir.glob("detect_*.json"))
        
        if detect_record_files:
            detect_record_path = detect_record_files[0]
            print(f"\n找到 detect record: {detect_record_path.name}")
            
            with open(detect_record_path, "r") as f:
                detect_record = json.load(f)
            
            # === [2/3] 验证结构 ===
            print("\n" + "=" * 60)
            print("[H-2] 验证 Detect Record 完整结构")
            print("=" * 60)
            
            print(f"\n✅ 已加载 detect_record.json，共 {len(detect_record)} 个字段")
            
            print("\n[顶级字段列表]")
            for key in sorted(detect_record.keys()):
                value_type = type(detect_record[key]).__name__
                print(f"  - {key}: {value_type}")
            
            print("\n[关键字段检查]")
            print(f"  ✅ paper_faithfulness: {type(detect_record.get('paper_faithfulness'))}")
            print(f"  ✅ decision: {type(detect_record.get('decision'))}")
            print(f"  ✅ fusion_result: {type(detect_record.get('fusion_result'))}")
            
            print("\n=== Detect Record 关键字段 ===")
            
            pf_detect = detect_record.get("paper_faithfulness", {})
            if isinstance(pf_detect, dict):
                print(f"\n[Paper Faithfulness 一致性检查]")
                pf_status = pf_detect.get("status", "UNKNOWN")
                print(f"  - status: {pf_status}")
                
                pf_mismatch = pf_detect.get("mismatch_reasons", [])
                if pf_mismatch:
                    print(f"  - mismatch_reasons: {pf_mismatch}")
                
                if pf_status in ["mismatch", "absent", "failed"]:
                    print(f"\n⚠️  Paper faithfulness 一致性问题: {pf_status}")
                elif pf_status == "ok":
                    print(f"\n✅ Paper faithfulness 一致性验证通过")
                else:
                    print(f"\n❓ 未知状态: {pf_status}")
            
            decision = detect_record.get("decision", {})
            if isinstance(decision, dict):
                is_watermarked = decision.get("is_watermarked")
                print(f"\n[检测决策]")
                print(f"  - is_watermarked: {is_watermarked}")
                print(f"  - decision_rule: {decision.get('rule', 'N/A')}")
            
            exec_report = detect_record.get("execution_report", {})
            if exec_report:
                print(f"\n[执行报告]")
                print(f"  - content_chain_status: {exec_report.get('content_chain_status', 'N/A')}")
                print(f"  - fusion_status: {exec_report.get('fusion_status', 'N/A')}")
            
            # === [3/3] 生成摘要对照表 ===
            print("\n" + "=" * 60)
            print("[H-3] Embed ↔ Detect 摘要对照表")
            print("=" * 60)
            
            try:
                embed_record_path_val = Path(RUN_ROOT) / "records" / "embed_record.json"
                
                if embed_record_path_val.exists():
                    with open(embed_record_path_val) as f:
                        embed_rec = json.load(f)
                    
                    print("\n[对照表] 核心摘要对齐验证")
                    print("-" * 80)
                    
                    digest_checks = []
                    
                    # Pipeline Fingerprint Digest
                    embed_pfp_digest = embed_rec.get("content_evidence", {}).get("pipeline_fingerprint_digest")
                    detect_pfp_digest = detect_rec.get("content_evidence_payload", {}).get("pipeline_fingerprint_digest")
                    pfp_match = "✅" if (embed_pfp_digest and detect_pfp_digest and embed_pfp_digest == detect_pfp_digest) else "❌"
                    digest_checks.append({
                        "摘要类型": "Pipeline Fingerprint",
                        "Embed": embed_pfp_digest[:16] + "..." if embed_pfp_digest else "<absent>",
                        "Detect": detect_pfp_digest[:16] + "..." if detect_pfp_digest else "<absent>",
                        "对齐": pfp_match
                    })
                    
                    # Trajectory Digest
                    embed_traj_digest = embed_rec.get("content_evidence", {}).get("trajectory_digest") or \
                                       embed_rec.get("content_evidence", {}).get("diffusion_trace_digest")
                    detect_traj_digest = detect_rec.get("content_evidence_payload", {}).get("trajectory_digest") or \
                                        detect_rec.get("content_evidence_payload", {}).get("diffusion_trace_digest")
                    traj_match = "✅" if (embed_traj_digest and detect_traj_digest and embed_traj_digest == detect_traj_digest) else "❌"
                    digest_checks.append({
                        "摘要类型": "Trajectory Digest",
                        "Embed": embed_traj_digest[:16] + "..." if embed_traj_digest else "<absent>",
                        "Detect": detect_traj_digest[:16] + "..." if detect_traj_digest else "<absent>",
                        "对齐": traj_match
                    })
                    
                    # Alignment Report Digest
                    embed_align_digest = embed_rec.get("content_evidence", {}).get("alignment_digest")
                    detect_align_digest = detect_rec.get("content_evidence_payload", {}).get("alignment_digest")
                    align_match = "✅" if (embed_align_digest and detect_align_digest and embed_align_digest == detect_align_digest) else "❌"
                    digest_checks.append({
                        "摘要类型": "Alignment Report",
                        "Embed": embed_align_digest[:16] + "..." if embed_align_digest else "<absent>",
                        "Detect": detect_align_digest[:16] + "..." if detect_align_digest else "<absent>",
                        "对齐": align_match
                    })
                    
                    # Injection Site Digest
                    embed_inj_digest = embed_rec.get("content_evidence", {}).get("injection_site_digest")
                    detect_inj_digest = detect_rec.get("content_evidence_payload", {}).get("injection_site_digest")
                    inj_match = "✅" if (embed_inj_digest and detect_inj_digest and embed_inj_digest == detect_inj_digest) else "❌"
                    digest_checks.append({
                        "摘要类型": "Injection Site Spec",
                        "Embed": embed_inj_digest[:16] + "..." if embed_inj_digest else "<absent>",
                        "Detect": detect_inj_digest[:16] + "..." if detect_inj_digest else "<absent>",
                        "对齐": inj_match
                    })
                    
                    df = pd.DataFrame(digest_checks)
                    print("\n" + df.to_string(index=False))
                    
                    align_count = sum(1 for c in digest_checks if c["对齐"] == "✅")
                    total_count = len(digest_checks)
                    
                    print(f"\n[统计] 摘要对齐: {align_count}/{total_count} 通过")
                    
                    if align_count == total_count:
                        print(f"\n✅ 完全对齐：所有摘要在 Embed 和 Detect 之间一致，证据链完整")
                    else:
                        print(f"\n⚠️  部分未对齐：{total_count - align_count} 个摘要不匹配或缺失")
                    
                    checked_digests = {
                        "generated_at": pd.Timestamp.now().isoformat(),
                        "checks": digest_checks,
                        "alignment_summary": {
                            "total": total_count,
                            "passed": align_count,
                            "failed": total_count - align_count,
                            "complete_alignment": align_count == total_count
                        }
                    }
                    
                    digest_report_path = Path(LOGS_DIR) / "checked_digests_alignment_report.json"
                    with open(digest_report_path, "w") as f:
                        json.dump(checked_digests, f, indent=2, ensure_ascii=False)
                    
                    print(f"\n✅ 摘要对照表已保存: {digest_report_path}")
                    
            except Exception as e:
                print(f"\n❌ 错误: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("\n⚠️  未找到 detect_record.json")


## I. 运行 Pytest 测试

In [ ]:
# Cell I: 运行 Pytest 测试
import subprocess
import os
from pathlib import Path

os.environ["CEG_WM_ENABLE_TORCH_TESTS"] = "1"

print("=" * 60)
print("[I] 运行 Pytest 测试")
print("=" * 60)
print("\n✅ 已启用 opt-in 测试: CEG_WM_ENABLE_TORCH_TESTS=1")

pytest_log = Path(LOGS_DIR) / "pytest.log"

print(f"\n日志输出: {pytest_log}")
print("\n开始执行 pytest（这可能需要几分钟）...\n")

cmd = ["python", "-m", "pytest", "tests/", "-v", "--tb=short"]

with open(pytest_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  Pytest 有失败用例，退出码: {proc.returncode}")
    print("\n最后 100 行日志:")
    !tail -100 {pytest_log}
else:
    print("\n✅ Pytest 全部通过")

print("\n=== 测试结果摘要 ===")
!grep -E "passed|failed|skipped|error" {pytest_log} | tail -5 || echo "未找到摘要"


## J. 运行严格审计与冻结签署

In [ ]:
# Cell J: 运行严格审计
import subprocess
import json
from pathlib import Path

print("=" * 60)
print("[J] 运行严格审计 (--strict)")
print("=" * 60)

audits_log = Path(LOGS_DIR) / "audits_strict.log"

print(f"\n日志输出: {audits_log}")
print("\n开始执行严格审计...\n")

cmd = ["python", "scripts/run_all_audits.py", "--strict"]

with open(audits_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    audit_output = []
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
        audit_output.append(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  审计脚本退出码: {proc.returncode}")

print("\n=== 审计结果摘要 ===")

try:
    audit_json_str = "".join(audit_output)
    json_start = audit_json_str.find("{")
    if json_start >= 0:
        audit_result = json.loads(audit_json_str[json_start:])
        
        summary = audit_result.get("summary", {})
        freeze_decision = summary.get("FreezeSignoffDecision", "UNKNOWN")
        blocking_reasons = summary.get("BlockingReasons", [])
        counts = summary.get("counts", {})
        
        print(f"\n[决策]")
        print(f"  - FreezeSignoffDecision: {freeze_decision}")
        print(f"  - BlockingReasons: {blocking_reasons if blocking_reasons else '无'}")
        
        print(f"\n[统计]")
        print(f"  - PASS: {counts.get('PASS', 0)}")
        print(f"  - FAIL: {counts.get('FAIL', 0)}")
        print(f"  - BLOCK_fails: {counts.get('BLOCK_fails', 0)}")
        
        if freeze_decision == "ALLOW_FREEZE":
            print(f"\n✅ 审计通过，允许冻结 (ALLOW_FREEZE)")
        else:
            print(f"\n❌ 审计未通过: {freeze_decision}")
            
            if counts.get('BLOCK_fails', 0) > 0:
                results = audit_result.get("results", [])
                for result in results:
                    if result.get("severity") == "BLOCK" and result.get("result") == "FAIL":
                        print(f"\n第一条 BLOCK 失败:")
                        print(f"  - audit_id: {result.get('audit_id', 'N/A')}")
                        print(f"  - rule: {result.get('rule', 'N/A')}")
                        print(f"  - impact: {result.get('impact', 'N/A')}")
                        break
        
        audit_result_path = Path(LOGS_DIR) / "audit_result.json"
        with open(audit_result_path, "w") as f:
            json.dump(audit_result, f, indent=2, ensure_ascii=False)
        print(f"\n审计结果已保存: {audit_result_path}")
        
except Exception as e:
    print(f"\n⚠️  无法解析审计结果 JSON: {e}")
    print("\n审计输出最后 50 行:")
    !tail -50 {audits_log}


## K. 生成对齐验收摘要

In [ ]:
# Cell K: 生成对齐验收摘要
import json
from pathlib import Path
from datetime import datetime

print("=" * 60)
print("[K] 生成对齐验收摘要")
print("=" * 60)

acceptance_summary = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "notebook_version": "v1.0",
    "run_root": str(RUN_ROOT),
    "model_id": MODEL_ID,
    "configuration": {
        "seed": SEED,
        "inference_steps": INFERENCE_NUM_STEPS,
        "guidance_scale": INFERENCE_GUIDANCE_SCALE,
        "prompt": INFERENCE_PROMPT,
        "enable_paper_faithfulness": ENABLE_PAPER_FAITHFULNESS
    },
    "embed_status": "unknown",
    "detect_status": "unknown",
    "pytest_status": "unknown",
    "audit_status": "unknown",
    "final_verdict": "UNKNOWN"
}

# Embed 状态
if embed_record_path and embed_record_path.exists():
    with open(embed_record_path, "r") as f:
        embed_rec = json.load(f)
    
    content_ev = embed_rec.get("content_evidence", {})
    alignment_status = content_ev.get("alignment_report", {}).get("overall_status", "UNKNOWN")
    
    acceptance_summary["embed_status"] = {
        "success": True,
        "alignment_overall_status": alignment_status,
        "record_path": str(embed_record_path)
    }
else:
    acceptance_summary["embed_status"] = {"success": False}

# Detect 状态
try:
    detect_records_dir = Path(detect_run_root) / "records"
    detect_rec_files = list(detect_records_dir.glob("detect_*.json"))
    if detect_rec_files:
        with open(detect_rec_files[0], "r") as f:
            detect_rec = json.load(f)
        
        pf_status = detect_rec.get("paper_faithfulness", {}).get("status", "UNKNOWN")
        decision = detect_rec.get("decision", {})
        
        acceptance_summary["detect_status"] = {
            "success": True,
            "paper_faithfulness_status": pf_status,
            "is_watermarked": decision.get("is_watermarked"),
            "record_path": str(detect_rec_files[0])
        }
    else:
        acceptance_summary["detect_status"] = {"success": False}
except:
    acceptance_summary["detect_status"] = {"success": False}

# Pytest 状态
pytest_log_path = Path(LOGS_DIR) / "pytest.log"
if pytest_log_path.exists():
    with open(pytest_log_path, "r") as f:
        pytest_output = f.read()
    
    import re
    match = re.search(r"(\d+) passed", pytest_output)
    passed_count = int(match.group(1)) if match else 0
    
    match_failed = re.search(r"(\d+) failed", pytest_output)
    failed_count = int(match_failed.group(1)) if match_failed else 0
    
    acceptance_summary["pytest_status"] = {
        "passed": passed_count,
        "failed": failed_count,
        "all_passed": failed_count == 0
    }
else:
    acceptance_summary["pytest_status"] = {"success": False}

# Audit 状态
audit_result_path = Path(LOGS_DIR) / "audit_result.json"
if audit_result_path.exists():
    with open(audit_result_path, "r") as f:
        audit_res = json.load(f)
    
    summary = audit_res.get("summary", {})
    acceptance_summary["audit_status"] = {
        "freeze_decision": summary.get("FreezeSignoffDecision", "UNKNOWN"),
        "blocking_reasons": summary.get("BlockingReasons", []),
        "pass_count": summary.get("counts", {}).get("PASS", 0),
        "fail_count": summary.get("counts", {}).get("FAIL", 0),
        "block_fails": summary.get("counts", {}).get("BLOCK_fails", 0)
    }
else:
    acceptance_summary["audit_status"] = {"success": False}

# 最终判定
conditions = [
    acceptance_summary["audit_status"].get("freeze_decision") == "ALLOW_FREEZE",
    acceptance_summary["pytest_status"].get("all_passed", False),
    acceptance_summary["embed_status"].get("alignment_overall_status") == "PASS",
    acceptance_summary["detect_status"].get("paper_faithfulness_status") not in ["mismatch", "absent", "failed"]
]

if all(conditions):
    acceptance_summary["final_verdict"] = "ACCEPT"
    acceptance_summary["verdict_reason"] = "所有验收条件满足"
else:
    acceptance_summary["final_verdict"] = "REJECT"
    failed_conditions = []
    if not conditions[0]:
        failed_conditions.append("审计未通过 ALLOW_FREEZE")
    if not conditions[1]:
        failed_conditions.append("pytest 有失败用例")
    if not conditions[2]:
        failed_conditions.append("embed alignment_report 未通过")
    if not conditions[3]:
        failed_conditions.append("detect paper_faithfulness 一致性问题")
    acceptance_summary["verdict_reason"] = "; ".join(failed_conditions)

summary_path = Path("/content/alignment_acceptance_summary.json")
with open(summary_path, "w") as f:
    json.dump(acceptance_summary, f, indent=2, ensure_ascii=False)

print(f"\n✅ 验收摘要已生成: {summary_path}")

print("\n" + "=" * 60)
print("最终验收判定")
print("=" * 60)
print(f"\n结果: {acceptance_summary['final_verdict']}")
print(f"原因: {acceptance_summary['verdict_reason']}")

if acceptance_summary['final_verdict'] == "ACCEPT":
    print("\n✅ 所有验收条件满足，证据包有效")
else:
    print("\n❌ 存在未满足的验收条件")

print(f"\n验收摘要详情:")
print(json.dumps(acceptance_summary, indent=2, ensure_ascii=False))


## L. 打包并下载证据包

In [ ]:
# Cell L: 打包并下载证据包
import shutil
from pathlib import Path
from google.colab import files

print("=" * 60)
print("[L] 打包证据包")
print("=" * 60)

artifacts_out = Path("/content/artifacts_out")
artifacts_out.mkdir(parents=True, exist_ok=True)

print("\n[1/4] 复制 run_root...")
run_root_copy = artifacts_out / "run_root"
if Path(RUN_ROOT).exists():
    shutil.copytree(RUN_ROOT, run_root_copy, dirs_exist_ok=True)
    print(f"  ✅ {RUN_ROOT} → {run_root_copy}")

try:
    if Path(detect_run_root).exists():
        detect_copy = artifacts_out / "run_root_detect"
        shutil.copytree(detect_run_root, detect_copy, dirs_exist_ok=True)
        print(f"  ✅ {detect_run_root} → {detect_copy}")
except:
    pass

print("\n[2/4] 复制 run_logs...")
logs_copy = artifacts_out / "run_logs"
if Path(LOGS_DIR).exists():
    shutil.copytree(LOGS_DIR, logs_copy, dirs_exist_ok=True)
    print(f"  ✅ {LOGS_DIR} → {logs_copy}")

print("\n[3/4] 复制验收摘要...")
summary_src = Path("/content/alignment_acceptance_summary.json")
if summary_src.exists():
    shutil.copy(summary_src, artifacts_out / "alignment_acceptance_summary.json")
    print(f"  ✅ 验收摘要已复制")

print("\n[4/4] 打包为 ZIP...")
bundle_zip = Path("/content/sd3_paper_faithfulness_run_bundle.zip")
shutil.make_archive(
    str(bundle_zip.with_suffix("")),
    'zip',
    artifacts_out
)

bundle_size_mb = bundle_zip.stat().st_size / (1024 * 1024)
print(f"  ✅ 打包完成: {bundle_zip.name} ({bundle_size_mb:.2f} MB)")

print("\n" + "=" * 60)
print("准备下载...")
print("=" * 60)

print("\n下载文件:")
files.download(str(bundle_zip))

print("\n✅ 下载完成！")
print("\n证据包内容:")
print("  - run_root/: embed 运行完整输出（records + artifacts + logs）")
print("  - run_root_detect/: detect 运行完整输出（如果执行成功）")
print("  - run_logs/: 所有执行日志（embed/detect/pytest/audits）")
print("  - alignment_acceptance_summary.json: 验收摘要")
print("\n请解压后检查验收摘要中的 final_verdict 字段。")


## 完成

### 验收标准

检查下载的 `alignment_acceptance_summary.json` 文件，确认以下条件：

1. ✅ `audit_status.freeze_decision` == `"ALLOW_FREEZE"`
2. ✅ `pytest_status.all_passed` == `true`
3. ✅ `embed_status.alignment_overall_status` == `"PASS"`
4. ✅ `detect_status.paper_faithfulness_status` 不为 `"mismatch"`/`"absent"`/`"failed"`

若所有条件满足，`final_verdict` 应为 `"ACCEPT"`。

### 故障排查

如遇到问题：

1. **模型下载失败**：检查 Cell C 是否正确配置 HF_TOKEN
2. **Embed 失败**：查看 `run_logs/embed.log` 最后 50 行
3. **对齐未通过**：检查 embed_record.json 中 `content_evidence.alignment_report.failures`
4. **审计 BLOCK**：查看 `run_logs/audit_result.json` 中第一条 BLOCK 的详细信息

### 模型说明

本 Notebook 使用 **Stable Diffusion 3.5 Medium**，采用 **DiT (Diffusion Transformer)** 架构，而非传统 UNet。
模型通过 `diffusers` 库的 `StableDiffusion3Pipeline` 加载。